# Notebook 04 - Statistical Analysis
running hypothesis tests on the cleaned data to back up our findings with numbers


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.tsa.seasonal import seasonal_decompose
import json

df = pd.read_csv('../data/processed/cleaned.csv')
results = {}
print("loaded", len(df), "rows")


### test 1 - independent t-test
are out of state drivers getting fined differently than ny drivers?

h0: average fine for ny plates = average fine for out of state plates

h1: they are different


In [ ]:
ny_fines = df[df['Is_OutOfState'] == 0]['Fine_Amount'].dropna()
oos_fines = df[df['Is_OutOfState'] == 1]['Fine_Amount'].dropna()
t_stat, p_val = stats.ttest_ind(ny_fines, oos_fines)
pooled_std = np.sqrt((ny_fines.std()**2 + oos_fines.std()**2) / 2)
cohens_d = (oos_fines.mean() - ny_fines.mean()) / pooled_std
results['t_test'] = {'t_stat': round(float(t_stat), 4), 'p_value': round(float(p_val), 6), 'cohens_d': round(float(cohens_d), 4)}
print("t-stat:", round(t_stat, 4), "p-value:", round(p_val, 6))
print("ny avg:", round(ny_fines.mean(), 2), "oos avg:", round(oos_fines.mean(), 2))


if p value is less than 0.05 we reject the null hypothesis. this tells us whether the city charges out of state drivers differently or if they just happen to commit different types of violations.


### test 2 - chi square test
is the type of violation independent of which borough it happens in?

h0: violation type and borough are independent

h1: they are dependent


In [ ]:
contingency = pd.crosstab(df['Violation_County'], df['Violation_Description'])
chi2, p, dof, expected = stats.chi2_contingency(contingency)
n = contingency.sum().sum()
k = min(contingency.shape) - 1
cramers_v = np.sqrt(chi2 / (n * k)) if k > 0 else 0
results['chi_square'] = {'chi2': round(float(chi2), 2), 'p_value': float(p), 'cramers_v': round(float(cramers_v), 4), 'dof': int(dof)}
print("chi2:", round(chi2, 2), "p-value:", p, "cramers v:", round(cramers_v, 4))


a significant result here means different boroughs have fundamentally different violation profiles. this is important because it means the city cannot use a one size fits all enforcement strategy.


### test 3 - one way anova with tukey hsd
do boroughs differ in their average fine amounts?

h0: all boroughs have the same mean fine

h1: at least one borough is different


In [ ]:
borough_fines = df[['Violation_County', 'Fine_Amount']].dropna()
groups = [g['Fine_Amount'].values for _, g in borough_fines.groupby('Violation_County')]
f_stat, p_anova = stats.f_oneway(*groups)
results['anova'] = {'f_stat': round(float(f_stat), 4), 'p_value': float(p_anova)}
print("f-stat:", round(f_stat, 4), "p-value:", p_anova)

tukey = pairwise_tukeyhsd(borough_fines['Fine_Amount'], borough_fines['Violation_County'], alpha=0.05)
results['tukey_significant_pairs'] = int(sum(tukey.reject))
print("significant pairwise differences:", sum(tukey.reject))


anova tells us if there is any difference between groups. tukey hsd then tells us exactly which boroughs differ from each other. this helps us identify which boroughs might need adjusted fine structures.


### test 4 - pearson correlation
is there a relationship between how old a car is and how many tickets it gets?

h0: no linear relationship

h1: there is a linear relationship


In [ ]:
vehicle_data = df.groupby('Plate_ID').agg(
    ticket_count=('Summons_Number', 'count'),
    avg_age=('Vehicle_Age', 'first')
).dropna()
corr, p_corr = stats.pearsonr(vehicle_data['avg_age'], vehicle_data['ticket_count'])
results['pearson'] = {'correlation': round(float(corr), 4), 'p_value': round(float(p_corr), 6)}
print("correlation:", round(corr, 4), "p-value:", round(p_corr, 6))


if the correlation is close to zero it means vehicle age has nothing to do with how many tickets someone gets. repeat offending is a behavioral issue not a demographic one.


### test 5 - time series decomposition
does ticket volume follow a weekly seasonal pattern?

h0: no seasonal pattern exists

h1: there is a seasonal component


In [ ]:
daily = df.groupby('Issue_Date')['Summons_Number'].count().sort_index()
daily.index = pd.to_datetime(daily.index)
daily = daily.resample('D').sum().fillna(0)
if len(daily) >= 14:
    decomp = seasonal_decompose(daily, model='additive', period=7)
    seasonal_var = float(np.var(decomp.seasonal.dropna()))
else:
    seasonal_var = 0.0
results['seasonal'] = {'seasonal_variance': round(seasonal_var, 4)}
print("seasonal variance:", round(seasonal_var, 4))


a high seasonal variance confirms that ticketing follows a predictable weekly cycle. enforcement drops on weekends which creates a gap that could be exploited by habitual violators.


### test 6 - linear regression (ols)
can we predict how much a fine will be based on vehicle characteristics?

h0: vehicle age, state, and repeat status do not predict fine amount

h1: at least one predictor is significant


In [ ]:
reg_df = df.dropna(subset=['Fine_Amount', 'Vehicle_Age', 'Is_OutOfState', 'Is_Repeat_Offender']).copy()
X = reg_df[['Vehicle_Age', 'Is_OutOfState', 'Is_Repeat_Offender']]
X = sm.add_constant(X)
y = reg_df['Fine_Amount']
model = sm.OLS(y, X).fit()
results['regression'] = {'r_squared': round(float(model.rsquared), 4), 'f_pvalue': float(model.f_pvalue)}
print("r-squared:", round(model.rsquared, 4))
print("f p-value:", model.f_pvalue)


a low r-squared means these variables alone cannot explain the variation in fines. this makes sense because the fine is mostly determined by the violation code itself not by who committed it. the penalty structure is rigid.


saving all results


In [ ]:
with open('../docs/statistical_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("all 6 tests completed and saved")
